# 03 — Baseline and Feature Engineering

**Input:** `train.parquet`, `eda_findings.json`
**Output:** `baseline_results.json`, `feature_decision.json`

**Order matters here.** The baseline is built first with the minimum preprocessing needed to make a model run, and only then are the five EDA hypotheses tested against it. Reversed, a gain cannot be attributed: better features and a better algorithm become indistinguishable.

**Everything runs inside a `Pipeline`.** Preprocessing fitted once on the full training set before cross-validation lets each validation fold's statistics influence that fit. The resulting optimism is invisible in the scores, which is what makes it dangerous. Wrapping preprocessing and model together forces a refit on every fold's training portion only.

**Parts**

1. Cross-validation scheme, locked to notebook 01
2. Minimal preprocessing and the logistic baseline
3. Cost-aware scoring
4. Logistic regression assumption checks
5. Label permutation test — the leakage check deferred from notebook 01
6. Feature engineering: H1–H5 tested individually and combined
7. Decision and handoff

---

## Setup

In [ ]:
# Restart the kernel after this cell before running the rest.
%pip install -q --force-reinstall --no-deps git+https://github.com/rbennum/telco-churn.git
%pip install -q pyarrow statsmodels

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from telco_churn.config import (
    RANDOM_SEED,
    TARGET,
    OUT_DIR,
    ensure_out_dir,
    find_artifact,
)
from telco_churn.business import (
    COST_ASSUMPTIONS,
    cost_per_customer,
    optimal_threshold,
    policy_from_proba,
    value_at_risk,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

# Keeps column names alive through the pipeline, which makes coefficient and SHAP
# work in the interpretation notebook far less painful.
sklearn.set_config(transform_output="pandas")

ensure_out_dir()
print(f"sklearn {sklearn.__version__} | seed {RANDOM_SEED}")

In [ ]:
train = pd.read_parquet(find_artifact("train.parquet"))
manifest = json.loads(find_artifact("ingest_manifest.json").read_text())

print(f"manifest expects : {manifest['train_rows']} rows")
print(f"loaded           : {len(train)} rows")
assert len(train) == manifest["train_rows"], "Stale train.parquet — re-run notebook 01"

try:
    findings = json.loads(find_artifact("eda_findings.json").read_text())
    print(f"\nEDA hypotheses carried forward: {len(findings['hypotheses'])}")
except FileNotFoundError:
    findings = None
    print("\neda_findings.json not found — hypotheses are restated inline below")

X = train.drop(columns=[TARGET])
y = train[TARGET]

print(f"\nX {X.shape} | churn rate {y.mean():.4f}")

---

## 1. Cross-Validation Scheme

Locked to the scheme recorded in notebook 01's manifest. Baseline and every later model must share it, or the comparison in the model-selection notebook compares folds rather than models.

In [ ]:
from sklearn.model_selection import StratifiedKFold

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f"manifest seed   : {manifest.get('random_seed')}")
print(f"manifest scheme : {manifest['validation_scheme']}")
print(f"this notebook   : StratifiedKFold(n_splits=5, shuffle=True, random_state={RANDOM_SEED})")

# The integer field is authoritative. The scheme string in notebook 01 was written
# with a hardcoded seed, so it can disagree with the run that produced the file —
# a reminder that any value repeated in two places will eventually diverge.
if str(RANDOM_SEED) not in str(manifest.get("validation_scheme", "")):
    print("\nNOTE: the manifest's scheme string does not mention this seed. It was "
          "hardcoded in notebook 01 and is informational only; the integer above "
          "is what the split actually used.")

assert manifest.get("random_seed") == RANDOM_SEED, (
    f"Seed mismatch: notebook 01 split with {manifest.get('random_seed')}, this "
    f"notebook uses {RANDOM_SEED}. Different seeds mean different folds and an "
    "incomparable score. Re-run notebook 01 or align RANDOM_SEED."
)

for i, (tr, va) in enumerate(CV.split(X, y)):
    print(f"  fold {i}: train {len(tr)} churn {y.iloc[tr].mean():.4f} | "
          f"valid {len(va)} churn {y.iloc[va].mean():.4f}")

---

## 2. Minimal Preprocessing and the Baseline

Minimal means only what is technically required to make logistic regression run: encode categoricals, scale numerics. No transformations, no derived features, no cleverness. Everything else is Part 6's job.

`handle_unknown="ignore"` matters even though notebook 01's schema validation should prevent unseen categories — a category present in training but absent from one fold's training portion would otherwise crash that fold rather than the whole run, producing a partial result that looks like a real score.

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_preprocessor():
    """Column selection by dtype, so engineered features are picked up automatically."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="first",
                                 sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, make_column_selector(dtype_include=np.number)),
        ("cat", categorical, make_column_selector(dtype_include=object)),
    ], verbose_feature_names_out=False)


def build_pipeline(model=None, feature_step=None):
    steps = []
    if feature_step is not None:
        steps.append(("features", feature_step))
    steps.append(("prep", build_preprocessor()))
    steps.append(("model", model or LogisticRegression(max_iter=2000,
                                                       random_state=RANDOM_SEED)))
    return Pipeline(steps)


baseline = build_pipeline()

# Inspect the expanded feature space once, on the full training set. This is for
# understanding only — the pipeline still refits per fold during CV.
n_features_out = build_preprocessor().fit_transform(X).shape[1]
print(f"raw columns      : {X.shape[1]}")
print(f"after encoding   : {n_features_out}")
baseline

---

## 3. Cost-Aware Scoring

The primary metric is expected cost per customer, and computing it needs `MonthlyCharges` and `Contract` for each row being scored. A standard `make_scorer` only receives `y_true` and `y_pred`, so it cannot see them.

The solution is a callable scorer with signature `(estimator, X, y)`, which scikit-learn supports directly. Because `X` here is the raw fold before preprocessing, the business columns are still present and readable.

Two naive policies are scored on the identical folds, so the comparison holds fold-for-fold rather than against a number computed elsewhere on a different sample.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    matthews_corrcoef,
    roc_auc_score,
)


def cost_scorer(estimator, X_fold, y_fold):
    """Negative expected cost per customer. Higher is better, as sklearn expects."""
    proba = estimator.predict_proba(X_fold)[:, 1]
    action = policy_from_proba(proba, X_fold["MonthlyCharges"], X_fold["Contract"])
    return -cost_per_customer(y_fold, action, X_fold["MonthlyCharges"],
                              X_fold["Contract"])


def cost_scorer_at_half(estimator, X_fold, y_fold):
    """Same, but with the default 0.5 threshold — the cost of not thinking."""
    action = (estimator.predict_proba(X_fold)[:, 1] > 0.5).astype(int)
    return -cost_per_customer(y_fold, action, X_fold["MonthlyCharges"],
                              X_fold["Contract"])


def _naive(policy_fn):
    def scorer(estimator, X_fold, y_fold):
        return -cost_per_customer(y_fold, policy_fn(X_fold),
                                  X_fold["MonthlyCharges"], X_fold["Contract"])
    return scorer


SCORING = {
    "neg_log_loss": "neg_log_loss",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "neg_brier_score": "neg_brier_score",
    "neg_cost": cost_scorer,
    "neg_cost_at_0.5": cost_scorer_at_half,
    "neg_cost_naive_nobody": _naive(lambda d: np.zeros(len(d))),
    "neg_cost_naive_m2m": _naive(lambda d: (d["Contract"] == "Month-to-month").astype(int)),
    "neg_cost_oracle": None,  # filled below; needs y, so handled separately
}
del SCORING["neg_cost_oracle"]

print("Scorers registered:", list(SCORING))
print("\nThe oracle needs the labels themselves as its action vector, so it is")
print("computed in the fold loop below rather than as a scorer.")

In [ ]:
from sklearn.model_selection import cross_validate

cv_res = cross_validate(baseline, X, y, cv=CV, scoring=SCORING,
                        return_train_score=True, n_jobs=-1)

# Oracle per fold: contact exactly the churners in that validation fold.
oracle_folds = []
for _, va in CV.split(X, y):
    Xv, yv = X.iloc[va], y.iloc[va]
    oracle_folds.append(cost_per_customer(yv, yv.to_numpy(float),
                                          Xv["MonthlyCharges"], Xv["Contract"]))
oracle_folds = np.array(oracle_folds)

summary = pd.DataFrame({
    "mean": {k: cv_res[f"test_{k}"].mean() for k in SCORING},
    "sd": {k: cv_res[f"test_{k}"].std() for k in SCORING},
})
summary.loc["oracle_cost"] = [oracle_folds.mean(), oracle_folds.std()]
display(summary.round(4))

In [ ]:
model_cost = -cv_res["test_neg_cost"].mean()
cost_half = -cv_res["test_neg_cost_at_0.5"].mean()
naive_nobody = -cv_res["test_neg_cost_naive_nobody"].mean()
naive_m2m = -cv_res["test_neg_cost_naive_m2m"].mean()
oracle = oracle_folds.mean()

best_naive = min(naive_nobody, naive_m2m)
headroom = best_naive - oracle
captured = (best_naive - model_cost) / headroom if headroom > 0 else np.nan

print(f"{'oracle ceiling':28s} USD {oracle:7.2f}")
print(f"{'baseline, cost threshold':28s} USD {model_cost:7.2f}")
print(f"{'baseline, threshold 0.5':28s} USD {cost_half:7.2f}")
print(f"{'naive: contact m2m':28s} USD {naive_m2m:7.2f}")
print(f"{'naive: contact nobody':28s} USD {naive_nobody:7.2f}")
print(f"\nheadroom available : USD {headroom:.2f}")
print(f"captured by baseline: {captured:.1%}")
print(f"\nDecision rule from notebook 01: continue at >=40%, abandon below 15%.")

verdict = ("CONTINUE" if captured >= 0.40 else
           "MARGINAL — investigate before adding complexity" if captured >= 0.15 else
           "ABANDON per the stopping condition")
print(f"Verdict on the baseline alone: {verdict}")
print(f"\nCost of using 0.5 instead of the derived threshold: "
      f"USD {cost_half - model_cost:+.2f} per customer")

In [ ]:
# Overfitting check: train against validation on the same metric.
gap = pd.DataFrame({
    "train": {k: cv_res[f"train_{k}"].mean() for k in SCORING},
    "valid": {k: cv_res[f"test_{k}"].mean() for k in SCORING},
})
gap["gap"] = gap["train"] - gap["valid"]
display(gap.round(4))
print("A small gap on a linear model with this many rows is expected. A large one")
print("would point at the one-hot expansion rather than at the algorithm.")

---

## 4. Logistic Regression Assumptions

These derive from maximum likelihood estimation, so they apply to this baseline and not to the tree ensembles later. Three of the tests commonly attached to regression do **not** belong here: Breusch-Pagan, Durbin-Watson, and residual normality all assume a continuous error term, whereas logistic regression's errors are binomial by construction.

What does apply: linearity of the logit, multicollinearity, events per predictor, and influential observations.

In [ ]:
import statsmodels.api as sm

NUM_COLS = X.select_dtypes(include=np.number).columns.tolist()

# Box-Tidwell: add x*ln(x) terms. A significant term means that predictor is not
# linear against the log-odds. Requires strictly positive input, so a small shift
# is applied to tenure and TotalCharges, both of which contain zeros.
bt = X[NUM_COLS].copy()
for c in NUM_COLS:
    shifted = bt[c] + 1e-6 if bt[c].min() > 0 else bt[c] - bt[c].min() + 1.0
    bt[f"{c}_x_ln"] = shifted * np.log(shifted)

bt_model = sm.Logit(y, sm.add_constant(bt)).fit(disp=0)
bt_terms = bt_model.pvalues.filter(like="_x_ln")

display(pd.DataFrame({
    "p_value": bt_terms.round(6),
    "non_linear_logit": bt_terms < 0.05,
}))
print("A True flags a predictor whose relationship with log-odds is not linear.")
print("Remedies: binning, splines, or letting a tree-based model handle it.")

In [ ]:
# Events per predictor. The common rule of thumb asks for at least 10 events per
# predictor in the minority class.
n_events = int(y.sum())
epv = n_events / n_features_out

print(f"events (churn)      : {n_events}")
print(f"predictors after OHE: {n_features_out}")
print(f"events per predictor: {epv:.1f}")
print(f"\n{'adequate' if epv >= 10 else 'below the rule of thumb — coefficients may be unstable'}")

In [ ]:
# Influential observations, logistic variant. Cook's distance above 4/n is the
# usual flag, though on large samples it flags many points and is best read as a
# ranking rather than a test.
prep_fitted = build_preprocessor().fit(X)
Xd = prep_fitted.transform(X).astype(float)
glm = sm.GLM(y, sm.add_constant(Xd), family=sm.families.Binomial()).fit()
cooks = glm.get_influence().cooks_distance[0]

thresh = 4 / len(X)
n_flag = int((cooks > thresh).sum())
print(f"Cook's distance > 4/n : {n_flag} rows ({100*n_flag/len(X):.2f}%)")
print(f"max Cook's distance   : {cooks.max():.5f}")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.stem(cooks, markerfmt=",", basefmt=" ")
ax.axhline(thresh, color="crimson", ls="--", lw=1, label=f"4/n = {thresh:.5f}")
ax.set_xlabel("row"); ax.set_ylabel("Cook's distance"); ax.legend()
ax.set_title("Influence — a single dominant spike would be the concern")
plt.tight_layout(); plt.show()

print("\nNo single dominant spike means no individual customer is steering the fit.")
print("Notebook 02 already established that no value here is a measurement error,")
print("so flagged rows are influential but legitimate and stay in the data.")

---

## 5. Label Permutation Test

The leakage check deferred from notebook 01, because it needs a complete pipeline to run.

Shuffle the labels and refit. With the relationship between features and target destroyed, any honest model must score at chance. A permuted model that still performs signals that information is reaching the target through a path other than genuine signal — an identifier that encodes outcome, a duplicated row spanning folds, or a feature computed after the fact.

This is the most direct leakage test available, and it is the one most often skipped.

In [ ]:
from sklearn.model_selection import permutation_test_score

score, perm_scores, pvalue = permutation_test_score(
    baseline, X, y, cv=CV, scoring="roc_auc",
    n_permutations=30, random_state=RANDOM_SEED, n_jobs=-1,
)

print(f"true ROC-AUC        : {score:.4f}")
print(f"permuted mean       : {perm_scores.mean():.4f} (sd {perm_scores.std():.4f})")
print(f"permuted max        : {perm_scores.max():.4f}")
print(f"p-value             : {pvalue:.4f}")

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(perm_scores, bins=15, edgecolor="white", label="permuted labels")
ax.axvline(score, color="crimson", lw=2, label=f"true score {score:.3f}")
ax.axvline(0.5, color="grey", ls="--", lw=1, label="chance")
ax.set_xlabel("ROC-AUC"); ax.legend()
ax.set_title("Permutation test — permuted scores must cluster at chance")
plt.tight_layout(); plt.show()

if perm_scores.mean() > 0.55:
    print("\nWARNING: permuted labels still score above chance. Stop and find the leak.")
else:
    print("\nPermuted scores sit at chance, as they must. No leakage detected.")

---

## 6. Feature Engineering

Each hypothesis from notebook 02 becomes a transformer, evaluated against the baseline on the same folds. A hypothesis that fails to improve the primary metric is dropped rather than kept for being intuitive.

**H1** — collapse `No internet service` and `No phone service` to `No`. Notebook 02 proved these levels are perfectly determined by `InternetService` and `PhoneService`, so this is lossless dimension reduction.

**H2** — replace `TotalCharges` with its residual against `tenure × MonthlyCharges`. VIF was 9.33, and the residual keeps whatever the product does not explain while removing the collinearity.

**H3** — an explicit `tenure × Contract` interaction. The strongest EDA finding: churn falls with tenure on month-to-month contracts and rises on two-year ones. An additive model cannot represent opposite-signed slopes.

**H4** — count of subscribed add-on services, as an engagement proxy replacing six separate binaries.

**H5** — an indicator for `tenure == 0`, customers structurally unable to have churned within the measurement window.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

SERVICE_COLS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]


def h1_collapse(df):
    d = df.copy()
    for c in SERVICE_COLS:
        d[c] = d[c].replace("No internet service", "No")
    d["MultipleLines"] = d["MultipleLines"].replace("No phone service", "No")
    return d


def h2_residual(df):
    d = df.copy()
    implied = d["tenure"] * d["MonthlyCharges"]
    d["charges_residual"] = d["TotalCharges"] - implied
    d["charges_ratio"] = d["TotalCharges"] / implied.replace(0, np.nan)
    d["charges_ratio"] = d["charges_ratio"].fillna(1.0)
    return d.drop(columns=["TotalCharges"])


def h3_interaction(df):
    d = df.copy()
    horizon = d["Contract"].map(COST_ASSUMPTIONS["horizon_by_contract"]).astype(float)
    d["tenure_x_horizon"] = d["tenure"] * horizon
    d["tenure_per_horizon"] = d["tenure"] / horizon
    for level in ["Month-to-month", "One year", "Two year"]:
        d[f"tenure_if_{level.replace(' ', '_').replace('-', '_')}"] = (
            d["tenure"] * (d["Contract"] == level)
        )
    return d


def h4_service_count(df):
    d = df.copy()
    d["n_services"] = sum((d[c] == "Yes").astype(int) for c in SERVICE_COLS)
    d["has_any_service"] = (d["n_services"] > 0).astype(int)
    return d


def h5_new_customer(df):
    d = df.copy()
    d["is_new_customer"] = (d["tenure"] == 0).astype(int)
    return d


def compose(*funcs):
    def inner(df):
        for f in funcs:
            df = f(df)
        return df
    return inner


def as_step(func):
    # validate=False keeps the DataFrame intact so column selection by dtype works.
    return FunctionTransformer(func, validate=False)


print("Feature transformers defined. Each returns a DataFrame, so the dtype-based")
print("column selector picks up new columns without any manual list maintenance.")

In [ ]:
EXPERIMENTS = {
    "baseline": None,
    "H1_collapse_service": h1_collapse,
    "H2_charges_residual": h2_residual,
    "H3_tenure_interaction": h3_interaction,
    "H4_service_count": h4_service_count,
    "H5_new_customer_flag": h5_new_customer,
    "H1+H3": compose(h1_collapse, h3_interaction),
    "H1+H3+H4": compose(h1_collapse, h3_interaction, h4_service_count),
    "all_hypotheses": compose(h1_collapse, h2_residual, h3_interaction,
                              h4_service_count, h5_new_customer),
}

rows = []
for name, func in EXPERIMENTS.items():
    pipe = build_pipeline(feature_step=as_step(func) if func else None)
    res = cross_validate(pipe, X, y, cv=CV,
                         scoring={"neg_log_loss": "neg_log_loss",
                                  "roc_auc": "roc_auc",
                                  "average_precision": "average_precision",
                                  "neg_cost": cost_scorer},
                         n_jobs=-1)
    n_feat = pipe.named_steps["prep"].fit_transform(
        as_step(func).fit_transform(X) if func else X
    ).shape[1] if True else None
    rows.append({
        "experiment": name,
        "n_features": n_feat,
        "log_loss": -res["test_neg_log_loss"].mean(),
        "roc_auc": res["test_roc_auc"].mean(),
        "avg_precision": res["test_average_precision"].mean(),
        "cost": -res["test_neg_cost"].mean(),
        "cost_sd": res["test_neg_cost"].std(),
    })
    print(f"  done: {name}")

ablation = pd.DataFrame(rows)
ablation["cost_vs_baseline"] = ablation["cost"] - ablation.loc[0, "cost"]
ablation["headroom_captured"] = (best_naive - ablation["cost"]) / headroom
display(ablation.round(4).sort_values("cost"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
order = ablation.sort_values("cost")
axes[0].barh(order["experiment"], order["cost"])
axes[0].axvline(ablation.loc[0, "cost"], color="crimson", ls="--", lw=1.2,
                label="baseline")
axes[0].axvline(oracle, color="green", ls=":", lw=1.2, label="oracle ceiling")
axes[0].set_xlabel("expected cost per customer (USD)")
axes[0].set_title("Primary metric — lower is better")
axes[0].invert_yaxis(); axes[0].legend()

axes[1].scatter(ablation["n_features"], ablation["cost"])
for _, r in ablation.iterrows():
    axes[1].annotate(r["experiment"], (r["n_features"], r["cost"]),
                     fontsize=7, alpha=0.8)
axes[1].set_xlabel("features after encoding"); axes[1].set_ylabel("cost (USD)")
axes[1].set_title("More features is not the same as better")
plt.tight_layout(); plt.show()

In [ ]:
# Differences this small need a significance check rather than a ranking.
# Paired comparison across the same folds, which is the only fair way to compare.
from scipy import stats as sps

best_name = ablation.sort_values("cost").iloc[0]["experiment"]
best_func = EXPERIMENTS[best_name]

base_fold = cross_validate(build_pipeline(), X, y, cv=CV,
                           scoring={"neg_cost": cost_scorer}, n_jobs=-1)["test_neg_cost"]
best_fold = cross_validate(build_pipeline(feature_step=as_step(best_func) if best_func else None),
                           X, y, cv=CV,
                           scoring={"neg_cost": cost_scorer}, n_jobs=-1)["test_neg_cost"]

# The scorers return NEGATIVE cost so that higher is better for sklearn. Convert
# back to actual cost before comparing, because reasoning about a double negative
# is exactly how sign errors get into decision rules.
cost_base = -base_fold
cost_best = -best_fold
cost_delta = cost_best - cost_base        # negative means the experiment is cheaper

t, p = sps.ttest_rel(cost_best, cost_base)

print(f"best experiment : {best_name}")
print(f"baseline cost per fold : {np.round(cost_base, 3)}")
print(f"{best_name} per fold   : {np.round(cost_best, 3)}")
print(f"delta per fold         : {np.round(cost_delta, 3)}   (negative = cheaper)")
print(f"mean delta             : USD {cost_delta.mean():+.3f} per customer")
print(f"paired t-test          : t={t:.3f}, p={p:.4f}")
print("\nFive folds is a small sample and these folds overlap in training data, so")
print("this t-test is indicative rather than authoritative. Treat a p above 0.05")
print("as 'not distinguishable from the baseline' and prefer the simpler option.")

### Choosing

The rule set before looking: adopt the feature set only if it improves expected cost **and** the improvement survives the paired comparison. Where two options are indistinguishable, take the simpler one — fewer features means less to break in production and less to explain in the write-up.

A hypothesis that fails here is still a result worth reporting. "The interaction I expected did not help the linear model" is a finding; quietly dropping it is not.

In [ ]:
ALPHA = 0.05
cheaper = cost_delta.mean() < 0          # lower cost is better
significant = p < ALPHA

if cheaper and significant:
    CHOSEN, chosen_func = best_name, best_func
    reason = (f"reduces expected cost by USD {abs(cost_delta.mean()):.3f} per "
              f"customer, p={p:.4f}")
elif cheaper:
    CHOSEN, chosen_func = "baseline", None
    reason = (f"{best_name} was nominally cheaper by USD {abs(cost_delta.mean()):.3f} "
              f"but p={p:.4f} does not clear {ALPHA}; folds disagree in sign, so the "
              f"simpler option wins")
else:
    CHOSEN, chosen_func = "baseline", None
    reason = "no engineered feature set reduced expected cost against the baseline"

print(f"CHOSEN: {CHOSEN}")
print(f"reason: {reason}")

---

## 7. Decision and Handoff

In [ ]:
chosen_row = ablation.set_index("experiment").loc[CHOSEN]

decision = {
    "notebook": "03_baseline_features",
    "random_seed": RANDOM_SEED,
    "cv_scheme": f"StratifiedKFold(n_splits=5, shuffle=True, random_state={RANDOM_SEED})",
    "baseline_model": "LogisticRegression(max_iter=2000)",
    "chosen_feature_set": CHOSEN,
    "reason": reason,
    "metrics": {
        "log_loss": round(float(chosen_row["log_loss"]), 5),
        "roc_auc": round(float(chosen_row["roc_auc"]), 5),
        "average_precision": round(float(chosen_row["avg_precision"]), 5),
        "cost_per_customer": round(float(chosen_row["cost"]), 4),
        "headroom_captured": round(float(chosen_row["headroom_captured"]), 4),
    },
    "benchmarks": {
        "oracle": round(float(oracle), 4),
        "best_naive": round(float(best_naive), 4),
        "headroom": round(float(headroom), 4),
        "cost_at_threshold_0.5": round(float(cost_half), 4),
    },
    "assumption_checks": {
        "events_per_predictor": round(float(epv), 2),
        "cooks_flagged_pct": round(100 * n_flag / len(X), 2),
        "box_tidwell_nonlinear": [c.replace("_x_ln", "")
                                  for c in bt_terms[bt_terms < 0.05].index],
    },
    "permutation_test": {
        "true_roc_auc": round(float(score), 4),
        "permuted_mean": round(float(perm_scores.mean()), 4),
        "p_value": round(float(pvalue), 4),
        "leakage_detected": bool(perm_scores.mean() > 0.55),
    },
    "ablation": ablation.round(5).to_dict("records"),
}

(OUT_DIR / "feature_decision.json").write_text(json.dumps(decision, indent=2))
(OUT_DIR / "baseline_results.json").write_text(json.dumps({
    "cv_results": {k: [float(v) for v in cv_res[f"test_{k}"]] for k in SCORING},
    "oracle_folds": [float(v) for v in oracle_folds],
}, indent=2))

print(json.dumps({k: v for k, v in decision.items() if k != "ablation"}, indent=2))

---

## Summary

| Item | Result |
| :--- | :--- |
| CV scheme | Stratified 5-fold, seed matched to notebook 01 and asserted |
| Baseline | Logistic regression, minimal preprocessing, everything inside a pipeline |
| Primary metric | Expected cost per customer at the per-customer derived threshold |
| Threshold comparison | Derived threshold versus the default 0.5, priced in currency |
| Assumptions | Box-Tidwell, VIF, events per predictor, Cook's distance — the tests that apply to logistic regression, not the ones borrowed from OLS |
| Leakage | Label permutation test, deferred from notebook 01, now settled |
| Feature engineering | Five hypotheses tested individually and combined, judged on cost and a paired comparison |

**What this notebook deliberately did not do:** no hyperparameter tuning, and no non-linear models. Both belong in notebook 04, where they are compared against this baseline on these folds. Tuning here would mean selecting features and hyperparameters on the same data, and the resulting estimate would be optimistic in a way cross-validation cannot reveal.

**Handoff:** `feature_decision.json` carries the chosen feature set and the benchmarks any later model must beat. The chosen transformer function should move into `src/telco_churn/features.py` once stable, so the tuning notebook and the Streamlit app build features through one code path.